## Lab 2: final challenges

__Вам предлагается решить задачу классификации сигналов или задачу классификации изображений. Или обе ;)__

__Выполнение этих заданий не является обязательным, но позитивно повлияет на вашу итоговую оценку. Успехов!__


### Part 4. HAR classification with raw data (2+ points)
__Disclaimer__: Это опциональная часть задания. Здесь придется экспериментировать, подбирать оптимальную структуру сети для решения задачи и активно искать подскзаки в сети.


Данное задание составлено на основе данного [поста](https://burakhimmetoglu.com/2017/08/22/time-series-classification-with-tensorflow/). С помощью вручную сгенерированных фичей и классических подходов задача распознования движений была решена с точностью 96%. 

Также будет полезным изучить [вот этот](https://github.com/healthDataScience/deep-learning-HAR), а так же [вот этот репозиторий](https://github.com/guillaume-chevalier/LSTM-Human-Activity-Recognition), где к данной задаче рассматривается несколько подходов.

In [126]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import math
import pylab
import warnings as w
import os

%matplotlib inline

In [127]:
import matplotlib
matplotlib.rcParams.update({'font.size':14})

Вернемся к задаче классификации движений на основе [данных](https://archive.ics.uci.edu/ml/datasets/Human+Activity+Recognition+Using+Smartphones) из репозитория UCI ([прямая ссылка на скачивание](https://archive.ics.uci.edu/ml/machine-learning-databases/00240/UCI%20HAR%20Dataset.zip)). 

В этот раз будем работать с исходными, а не предобработанными данными. Данные представляют собой сигналы с гироскопа и акселерометра, закрепленного на теле человека. Каждому семплу соотвествует 9 связанных временных рядов.

В начале приведена визуализация данных на основе PCA над вручную сгенерированными признаками. Для отрисовки графиков (цвет и легенда) нам также понадобятся метки классов.

In [ ]:
X_train_with_engineered_features = np.genfromtxt(os.path.join("UCI HAR Dataset", "train", "X_train.txt"))
y_train = np.genfromtxt(os.path.join("UCI HAR Dataset", "train", "y_train.txt"))

y_train_list = list(y_train)
X_unique = np.array([X_train_with_engineered_features[y_train_list.index(l)]
                             for l in sorted(list(set(y_train)))])

legend_labels = ["WALKING", "WALKING.UP", "WALKING.DOWN", "SITTING", "STANDING", "LAYING"]
colors_list = ['red', 'blue', 'green', 'orange', 'cyan', 'magenta']
mapped_colors = [colors_list[int(i)-1] for i in y_train]

from sklearn.decomposition import PCA
pca = PCA()

X_train_pca = pca.fit_transform(X_train_with_engineered_features)

plt.figure(figsize=(15,10))
pylab.scatter(X_train_pca[:, 0], X_train_pca[:, 1],
             c=mapped_colors)
plt.grid()
for idx, x in enumerate(pca.transform(X_unique)):
    plt.scatter(x[0], 
                x[1], 
                c=colors_list[idx], 
                label=legend_labels[idx])
plt.xlabel('First principal component')
plt.ylabel('Second principal component')
plt.legend()

#### Предобработка данных
Предобработка сделана за нас автором [данного репозитория](https://github.com/guillaume-chevalier/LSTM-Human-Activity-Recognition). Будьте осторожны с путями.

In [ ]:
# Useful Constants

# Those are separate normalised input features for the neural network
INPUT_SIGNAL_TYPES = [
    "body_acc_x_",
    "body_acc_y_",
    "body_acc_z_",
    "body_gyro_x_",
    "body_gyro_y_",
    "body_gyro_z_",
    "total_acc_x_",
    "total_acc_y_",
    "total_acc_z_"
]

# Output classes to learn how to classify
LABELS = [
    "WALKING", 
    "WALKING_UPSTAIRS", 
    "WALKING_DOWNSTAIRS", 
    "SITTING", 
    "STANDING", 
    "LAYING"
]

DATA_PATH = "./"

DATASET_PATH = DATA_PATH + "UCI HAR Dataset/"
print("\n" + "Dataset is now located at: " + DATASET_PATH)

TRAIN = "train/"
TEST = "test/"


# Load "X" (the neural network's training and testing inputs)

def load_X(X_signals_paths):
    X_signals = []
    
    for signal_type_path in X_signals_paths:
        file = open(signal_type_path, 'r')
        # Read dataset from disk, dealing with text files' syntax
        X_signals.append(
            [np.array(serie, dtype=np.float32) for serie in [
                row.replace('  ', ' ').strip().split(' ') for row in file
            ]]
        )
        file.close()
    
    return np.transpose(np.array(X_signals), (1, 2, 0))

X_train_signals_paths = [
    os.path.join(*[DATASET_PATH, TRAIN, "Inertial Signals/", signal+"train.txt"]) for signal in INPUT_SIGNAL_TYPES
]
X_test_signals_paths = [
    os.path.join(*[DATASET_PATH, TEST, "Inertial Signals/", signal+"test.txt"]) for signal in INPUT_SIGNAL_TYPES
]

X_train = load_X(X_train_signals_paths)
X_test = load_X(X_test_signals_paths)


# Load "y" (the neural network's training and testing outputs)

def load_y(y_path):
    file = open(y_path, 'r')
    # Read dataset from disk, dealing with text file's syntax
    y_ = np.array(
        [elem for elem in [
            row.replace('  ', ' ').strip().split(' ') for row in file
        ]], 
        dtype=np.int32
    )
    file.close()
    
    # Substract 1 to each output class for friendly 0-based indexing 
    return y_ - 1

y_train_path = os.path.join(DATASET_PATH, TRAIN, "y_train.txt")
y_test_path = os.path.join(DATASET_PATH, TEST, "y_test.txt")

y_train = load_y(y_train_path)
y_test = load_y(y_test_path)

In [ ]:
# Input Data 

training_data_count = len(X_train)  # 7352 training series (with 50% overlap between each serie)
test_data_count = len(X_test)  # 2947 testing series
n_steps = len(X_train[0])  # 128 timesteps per series
n_input = len(X_train[0][0])  # 9 input parameters per timestep


# LSTM Neural Network's internal structure

n_hidden = 32 # Hidden layer num of features
n_classes = 6 # Total classes (should go up, or should go down)


# Some debugging info

print("Some useful info to get an insight on dataset's shape and normalisation:")
print("(X shape, y shape, every X's mean, every X's standard deviation)")
print(X_test.shape, y_test.shape, np.mean(X_test), np.std(X_test))
print("The dataset is therefore properly normalised, as expected, but not yet one-hot encoded.")

#### Построение сети и эксперименты. (100% +)

__Ваша задача - построить сеть, которая решит задачу классификации с точностью (`accuracy`) не менее 86%.__
Разбалловка следующая:
* $=$86% - 2 points
* $>=$89% - 2.5 points
* $>=$91% - 3 points


__Warning!__ В сети существует несколько решений данной задачи с использованием различных фреймворков. При проверке это будет учитываться, так что свое решение нужно будет объяснить. Пожалуйста, не копируйте бездумно код, такие задания будут оценены 0 баллов. Если задача не решается - можете обратиться к заданию по классификации изображений.

После выполнения задания заполните небольшой отчет об экспериментах вида "Я пробовал(а) ... подходы и получил(а) ... результаты. Наконец, после N+1 чашки кофе/бессонной ночи у меня получилось, и весь секрет был в ..."

In [ ]:
# Your experiments here

### Part 5. Dogs classification (2+ points)
__Disclaimer__: Это опциональная часть задания. Здесь придется экспериментировать, подбирать оптимальную структуру сети для решения задачи и активно искать подскзаки в сети.

Предлагаем вам решить задачу классификации пород собак. Вы можете обучить сеть с нуля или же воспользоваться методом fine-tuning'а. Полезная ссылка на [предобученные модели](https://pytorch.org/docs/stable/torchvision/models.html).

Данные можно скачать [отсюда](https://www.dropbox.com/s/vgqpz2f1lolxmlv/data.zip?dl=0). Датасет представлен 50 классами пород собак, которые можно найти в папке train в соответствующих директориях. При сдаче данной части задания вместе с ноутбуком необходимо отправить .csv-файл с предсказаниями классов тестовой выборки в формате: <имя изображения>,<метка класса> по одному объекту на строку. Ниже приведите код ваших экспериментов и короткий вывод по их результатам.

Будут оцениваться качество классификации (accuracy) на тестовой выборке (2 балла) и проведенные эксперименты (1 балл).
Разбалловка следующая:
* $>=$93% - 2 points
* $>=$84% - 1.5 points
* $>=$70% - 0.75 points

In [107]:
# !pip install lightning==2.5.5 torchmetrics==1.8.2

In [185]:
import PIL.Image
import numpy as np
import torch
from torch import nn
import torchvision
import torchvision.transforms.v2 as T
import torch.nn.functional as F
from torch.utils import data
from torchvision import datasets, transforms
import torchmetrics
import lightning as L
from lightning.pytorch.loggers import TensorBoardLogger

In [186]:
DEVICE = torch.device("cuda")

In [187]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
NETWORK_SIZE = (224, 224)

In [188]:
test_transforms = T.Compose(
    [
        T.Resize(size=NETWORK_SIZE),
        T.ToTensor(),  
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]
)

train_transforms = T.Compose([
    T.RandomResizedCrop(NETWORK_SIZE, scale=(0.8, 1.0), ratio=(0.9, 1.1)),

    T.RandomAffine(
        degrees=15,
        translate=(0.05, 0.05),
        scale=(1.0, 1.05)
    ),

    T.ColorJitter(
        brightness=0.3,
        contrast=0.3,
        hue=0.06
    ),

    T.RandomHorizontalFlip(p=0.3),

    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

/media/alexander/D/PycharmProjects/ml-course-mipt/venv/lib/python3.12/site-packages/torchvision/transforms/v2/_deprecated.py:42: UserWarning: The transform `ToTensor()` is deprecated and will be removed in a future release. Instead, please use `v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])`.Output is equivalent up to float precision.
  warnings.warn(


In [189]:
def get_resnet_torchvision(num_classes: int):
    weights = torchvision.models.ResNet50_Weights.IMAGENET1K_V1
    model = torchvision.models.resnet50(weights=weights)

    for name, param in model.named_parameters():
        param.requires_grad = False

    classifier = torch.nn.Sequential(
        torch.nn.Linear(2048, 1024),
        torch.nn.ReLU(),
        torch.nn.Linear(1024, num_classes),
    )
    model.fc = classifier

    for name, param in model.named_parameters():
        if name.startswith("layer4"):
            param.requires_grad = True

    return model

In [190]:
class ResNetBasedClassifier(L.LightningModule):
    def __init__(
        self,
        *,
        num_classes: int,
        lr=1e-4,
        **kwargs,
    ):
        self.num_classes = num_classes
        super().__init__(**kwargs)
        self.lr = lr
        self.model = self.get_model()
        self.loss_fn = nn.CrossEntropyLoss(label_smoothing=0.5)
        self.accuracy = torchmetrics.classification.Accuracy(
            task="multiclass",
            num_classes=self.num_classes,
        )

    def get_model(self):
        return get_resnet_torchvision(self.num_classes)

    def configure_optimizers(self):
        return torch.optim.Adam(self.model.parameters(), lr=self.lr, weight_decay=1e-3)

    def training_step(self, batch):
        return self._step(batch, "train")

    def validation_step(self, batch):
        return self._step(batch, "valid")

    def _step(self, batch, kind):
        x, y = batch
        p = self.model(x)

        loss = self.loss_fn(p, y)
        accs = self.accuracy(p.argmax(axis=-1), y)

        return self._log_metrics(loss, accs, kind)

    def _log_metrics(self, loss, accs, kind):
        metrics = {}
        if loss is not None:
            metrics[f"{kind}_loss"] = loss
        if accs is not None:
            metrics[f"{kind}_accs"] = accs
        self.log_dict(
            metrics,
            prog_bar=True,
            logger=True,
            on_step=kind == "train",
            on_epoch=True,
        )
        return loss

In [191]:
def train_classifier(train_dataloader, test_dataloader, num_labels, num_epochs):
    model = ResNetBasedClassifier(num_classes=num_labels).to(DEVICE)

    callbacks = [
        L.pytorch.callbacks.TQDMProgressBar(leave=True),
    ]

    logger = TensorBoardLogger("tb_logs", name="dogs")
    
    trainer = L.Trainer(
        callbacks=callbacks, max_epochs=num_epochs, logger=logger, enable_checkpointing=False
    )
    
    # trainer = L.Trainer(max_epochs=num_epochs, logger=False, enable_checkpointing=False)

    trainer.fit(model, train_dataloader, test_dataloader)

    return model.model

In [192]:
data_dir = 'data/train'
dataset = datasets.ImageFolder(root=data_dir)

In [193]:
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

In [194]:
train_subset, test_subset = torch.utils.data.random_split(dataset, [train_size, test_size])

In [195]:
train_dataset = torch.utils.data.Subset(
    datasets.ImageFolder('data/train', transform=train_transforms),
    train_subset.indices
)

test_dataset = torch.utils.data.Subset(
    datasets.ImageFolder('data/train', transform=test_transforms),
    test_subset.indices
)

In [196]:
train_dataloader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)
test_dataloader = torch.utils.data.DataLoader(test_dataset, batch_size=64, shuffle=False)

In [120]:
# %load_ext tensorboard
# %tensorboard --logdir tb_logs

In [197]:
model = train_classifier(train_dataloader, test_dataloader, len(dataset.classes), 8)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name     | Type               | Params | Mode 
--------------------------------------------------------
0 | model    | ResNet             | 25.7 M | train
1 | loss_fn  | CrossEntropyLoss   | 0      | train
2 | accuracy | MulticlassAccuracy | 0      | train
--------------------------------------------------------
17.1 M    Trainable params
8.5 M     Non-trainable params
25.7 M    Total params
102.630   Total estimated model params size (MB)
156       Modules in train mode
0         Modules in eval mode


Training: |                                                                                                                         | 0/? [00:00<?, ?it/s]
Epoch 0: 100%|█████████████████████████████████████████████████████| 90/90 [00:53<00:00,  1.69it/s, v_num=0, train_loss_step=2.900, train_accs_step=0.861]
Validation: |                                                                                                                       | 0/? [00:00<?, ?it/s]
Validation: |                                                                                                                       | 0/? [00:00<?, ?it/s]
Validation DataLoader 0: 100%|████████████████████████████████████████████████████████████████████████████████████████████| 23/23 [00:06<00:00,  3.57it/s]
Epoch 0: 100%|█| 90/90 [01:00<00:00,  1.50it/s, v_num=0, train_loss_step=2.900, train_accs_step=0.861, valid_loss=2.890, valid_accs=0.838, train_loss_epoc
Epoch 1: 100%|█| 90/90 [00:49<00:00,  1.81it/s, v_num=0, train_loss_st

`Trainer.fit` stopped: `max_epochs=8` reached.


In [198]:
sd = model.state_dict()
torch.save(sd,"dogs_model.pt")

In [199]:
test_dir = 'data/test'

In [201]:
idx_to_class = {v:k for k, v in dataset.class_to_idx.items()}

In [204]:
model.eval()

files = [file for file in os.listdir(test_dir)]
print(len(files))
res = {}
for i, file in enumerate(files, 1):
    if i % 100 == 0:
        print(f"{i}/{len(files)}")
    img_path = f"{test_dir}/{file}"
    image = PIL.Image.open(img_path).convert("RGB")
    img = test_transforms(image)
    img = torch.unsqueeze(img, 0)
    
    y_pred = model(img)
    idx = F.softmax(y_pred, dim=1).argmax(axis=-1).tolist()[0]
    label = idx_to_class[idx]
    res[file] = label

1503
100/1503
200/1503
300/1503
400/1503
500/1503
600/1503
700/1503
800/1503
900/1503
1000/1503
1100/1503
1200/1503
1300/1503
1400/1503
1500/1503


In [148]:
import pandas as pd

In [205]:
test_res = pd.DataFrame.from_dict(res, orient='index').reset_index()

In [206]:
test_res.columns = ['file', 'label']

In [207]:
test_res.head()

,file,label
0,304.jpeg,32
1,980.jpeg,15
2,981.jpeg,46
3,982.jpeg,14
4,983.jpeg,17


In [208]:
test_res.to_csv('test_res.csv', index=False)

In [173]:
test_res.sort_values(by='label')

,file,label
234,281.jpeg,13
1348,867.jpeg,13
1035,735.jpeg,13
158,368.jpeg,13
268,1109.jpeg,13
...,...,...
501,450.jpeg,30
500,45.jpeg,30
499,449.jpeg,30
508,933.jpeg,30
